# Compile single-cell FACS into minimal dataframe

This script compiles the raw FACS results from each well into one dataframe and writes it out as a compressed parquet file. 

In [36]:
from glob import glob
from os import stat
import pandas as pd
import numpy as np

In [37]:
dualipa_inputs = "../1_inputs"
dualipa_outputs = "../3_outputs"

In [ ]:
# Download raw FACS data from Zenodo
bash_file = f"{dualipa_inputs}/download_raw_FACS.sh"
os.system(f"bash {bash_file}")

In [38]:
# Compile all data into single df
facs_data = f"{dualipa_inputs}/raw_inputs/FACS_single_cell_data_mCherry_positive_threshold_500/*"
all_df = []
for file_name in glob(facs_data):
    pla_id = file_name.split('/')[-1].split('_')[-3]
    well = file_name.split('/')[-1].split('_')[-2]
    if stat(file_name).st_size > 0:
        df = pd.read_csv(file_name,sep='\t',header=None).rename(columns={0:'GFP',1:'mCherry'})
        df['pla'] = int(pla_id)
        df['well'] = well
        all_df.append(df)

pDEST_DUAL_df = pd.concat(all_df)

In [39]:
# Replace zeros with minimum non-zero value to avoid undefined after taking log
min_nonzero_val = pDEST_DUAL_df[pDEST_DUAL_df["GFP"] > 0]["GFP"].min()
pDEST_DUAL_df.loc[pDEST_DUAL_df["GFP"] == 0, "GFP"] = min_nonzero_val

In [40]:
# Calculate ratios and log10 transformations
pDEST_DUAL_df['GFP_mCherry_ratio'] = np.log2(pDEST_DUAL_df.GFP / pDEST_DUAL_df.mCherry)
pDEST_DUAL_df['GFP_log10'] = pDEST_DUAL_df.GFP.apply(np.log10)
pDEST_DUAL_df['mCherry_log10'] = pDEST_DUAL_df.mCherry.apply(np.log10)

In [41]:
# Merge with minimal metadata
pdest_layout_df = pd.read_csv(f"{dualipa_outputs}/dualipa_experimental_layout.csv", index_col=0)
pDEST_DUAL_df = pDEST_DUAL_df.merge(pdest_layout_df, left_on=['pla','well'], right_on=['dest_pla_id','dest_well'])
pDEST_DUAL_df["coordinates"] = pDEST_DUAL_df["dest_pla_id"].astype(str) + "_" + pDEST_DUAL_df["well"].astype(str)

In [43]:
# Add cell count for each well
n_cell_d = dict(pDEST_DUAL_df.coordinates.value_counts())
pDEST_DUAL_df['n_cells'] = pDEST_DUAL_df.coordinates.map(n_cell_d)

In [45]:
# write out single-cell data
pDEST_DUAL_df.to_parquet(f"{dualipa_outputs}/facs_single_cell.parquet", compression="zstd")